# 1. Random Sampling and Sample Bias

**Population** adalah keseluruhan kelompok yang ingin dipahami, sedangkan **sample** adalah subset yang digunakan untuk memperoleh informasi tentang population.

**Simple random sample** diperoleh melalui random sampling tanpa stratifikasi.

**Sample bias** terjadi ketika sample salah merepresentasikan population secara sistematis.

### Size Versus Quality

Sample yang lebih kecil tetapi dipilih dengan baik dapat lebih berguna daripada dataset yang sangat besar tetapi biased atau sulit diperiksa. Big data tetap penting untuk kasus tertentu, misalnya data yang sangat sparse.

### Sample Mean Versus Population Mean

- $\bar{x}$ = sample mean
- $\mu$ = population mean

Sample diamati langsung, sedangkan population mean sering diinferensikan dari sample.

# 2. Sampling Distribution of a Statistic

**Sampling distribution** adalah distribusi nilai suatu sample statistic yang diperoleh dari banyak sample.

- **data distribution**: distribusi individual data points,
- **sampling distribution**: distribusi sample statistic.

Sampling distribution dari mean cenderung lebih regular dan lebih sempit daripada data distribution ketika sample size meningkat.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.utils import resample

loans_income = pd.read_csv('loans_income.csv').squeeze()

In [ ]:
sample_data = pd.DataFrame({
    'income': loans_income.sample(1000),
    'type': 'Data',
})
sample_mean_05 = pd.DataFrame({
    'income': [loans_income.sample(5).mean() for _ in range(1000)],
    'type': 'Mean of 5',
})
sample_mean_20 = pd.DataFrame({
    'income': [loans_income.sample(20).mean() for _ in range(1000)],
    'type': 'Mean of 20',
})
results = pd.concat([sample_data, sample_mean_05, sample_mean_20])

In [ ]:
g = sns.FacetGrid(results, col='type', col_wrap=1, height=2, aspect=2)
g.map(plt.hist, 'income', range=[0, 200000], bins=40)
g.set_axis_labels('Income', 'Count')
g.set_titles('{col_name}')

# 3. Central Limit Theorem

**Central Limit Theorem (CLT)** menyatakan bahwa mean dari banyak sample akan cenderung menyerupai bell-shaped normal curve ketika sample size cukup besar dan penyimpangan population dari normalitas tidak terlalu besar.

CLT membantu menjelaskan mengapa pendekatan normal dapat digunakan untuk sampling distributions.

# 4. Standard Error

**Standard error (SE)** mengukur variability sebuah sample statistic antar-sample.

Untuk sample mean:

$$SE = \frac{s}{\sqrt{n}}$$

**Square-root of n rule**: untuk mengurangi SE menjadi setengah, sample size perlu dinaikkan sekitar empat kali.

Jangan tertukar:
- standard deviation → variability individual data points,
- standard error → variability sample statistic.

In [ ]:
# Formula standard error of the sample mean
se = loans_income.std() / (len(loans_income) ** 0.5)
se

# 5. The Bootstrap

**Bootstrap** adalah pengambilan sample berulang **with replacement dari observed data** untuk mengestimasi sampling distribution.

Algoritmanya:
1. draw value,
2. replace value,
3. ulangi hingga sample size $n$,
4. hitung statistic,
5. ulangi $R$ kali,
6. gunakan hasil untuk standard error, histogram/boxplot, atau confidence interval.

Bootstrap tidak menciptakan data baru dan tidak memperbaiki sample size yang kecil.

In [ ]:
results = []
for nrepeat in range(1000):
    sample = resample(loans_income)
    results.append(sample.median())
results = pd.Series(results)

print('Bootstrap Statistics:')
print(f'original: {loans_income.median()}')
print(f'bias: {results.mean() - loans_income.median()}')
print(f'std. error: {results.std()}')

In [ ]:
ax = results.plot.hist(bins=30)
ax.set_xlabel('Bootstrap median')
ax.set_ylabel('Frequency')

### Multivariate Bootstrap

Bootstrap juga dapat diterapkan pada multivariate data dengan **row sebagai unit sampling**.

# 6. Confidence Intervals

Confidence interval menyajikan estimate sebagai **range**, bukan satu angka.

Confidence level adalah coverage level yang diharapkan dari prosedur interval jika prosedur yang sama diulang.

### Bootstrap confidence interval
1. Draw sample size $n$ with replacement.
2. Hitung statistic.
3. Ulangi $R$ kali.
4. Untuk confidence level $x\%$, trim $(100-x)/2\%$ dari masing-masing ekor.
5. Titik trim menjadi interval endpoints.

Semakin tinggi confidence level → interval lebih lebar.
Semakin kecil sample → interval lebih lebar.

# 7. Normal Distribution

Normal distribution berbentuk bell curve dengan parameter mean $\mu$ dan standard deviation $\sigma$.

Dalam normal distribution:
- sekitar 68% berada dalam ±1 SD,
- sekitar 95% berada dalam ±2 SD.



# 8. Standard Normal and QQ-Plots

**Standard normal** memiliki mean 0 dan standard deviation 1.

Standardization:

$$z = \frac{x-\bar{x}}{s}$$

QQ-Plot digunakan untuk melihat kedekatan sample terhadap specified distribution. Jika titik mengikuti garis diagonal, sample mendekati distribution tersebut.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
norm_sample = stats.norm.rvs(size=100)
stats.probplot(norm_sample, plot=ax)

# 9. Long-Tailed Distributions

**Tail** adalah bagian panjang dan sempit dari frequency distribution tempat extreme values muncul dengan frequency rendah.

**Skew** terjadi ketika salah satu tail lebih panjang daripada tail lainnya.


In [ ]:
sp500_px = pd.read_csv('sp500_data.csv.gz', index_col=0)

nflx = sp500_px.NFLX
nflx = np.diff(np.log(nflx[nflx > 0]))

fig, ax = plt.subplots(figsize=(4, 4))
stats.probplot(nflx, plot=ax)

# 10. Student's t-Distribution

Student's t-distribution menyerupai normal distribution tetapi memiliki tails yang lebih tebal dan panjang.

t-distribution merupakan keluarga distribution yang berbeda menurut **degrees of freedom**. Semakin besar sample size/degrees of freedom, bentuknya semakin mendekati normal.

# 11. Binomial Distribution

Binomial outcomes adalah **yes/no**, **0/1**, atau binary outcomes.

Binomial distribution menggambarkan jumlah successes $x$ dalam $n$ trials dengan probability success $p$.

$$P(X=x)=\binom{n}{x}p^x(1-p)^{n-x}$$

Mean:

$$E(X)=np$$

Variance:

$$Var(X)=np(1-p)$$

In [ ]:
stats.binom.pmf(2, n=5, p=0.1)

In [ ]:
stats.binom.cdf(2, n=5, p=0.1)

### Interpretasi

`pmf` memberikan probability **exactly** sejumlah successes.

`cdf` memberikan probability sejumlah successes **atau kurang**.


# 12. Chi-Square Distribution

Chi-square distribution digunakan untuk mengukur **departure from expectation**, terutama untuk category counts.

Chi-square statistic:

$$\chi^2=\sum\frac{(Observed-Expected)^2}{Expected}$$

Semakin besar perbedaan observed dan expected, semakin besar statistic.


# 13. F-Distribution

F-distribution digunakan pada eksperimen dan linear models dengan measured/continuous data.

F-statistic membandingkan variability antar group means dengan variability di dalam group:

$$F=\frac{\text{variation between groups}}
{\text{variation within groups}}$$


# 14. Poisson and Related Distributions

Poisson dan related distributions digunakan untuk proses yang menghasilkan events secara random dengan overall rate.

**Lambda ($\lambda$)** = event rate per unit time atau space.

## Poisson Distribution

Poisson distribution adalah distribution dari jumlah events pada unit waktu atau ruang.

Mean:

$$E(X)=\lambda$$

Variance:

$$Var(X)=\lambda$$


In [ ]:
stats.poisson.rvs(2, size=100)

## Exponential Distribution

Exponential distribution digunakan untuk **time/distance between events**, misalnya waktu antara website visits atau cars arriving.

## Weibull Distribution

Weibull adalah generalized version dari exponential distribution, dengan event rate yang dapat berubah sepanjang waktu.
